# Intelligent Emergency Dispatch System
### Data Structures & Algorithms Project

**Objective:**
To build a system that efficiently assigns ambulances to emergency incidents in a city. The goal is to minimize the response time (ETA) by selecting the optimal ambulance based on location and availability.

**Key Algorithms Used:**
1.  **Graph Representation (Adjacency List):** The city is modeled as a graph where intersections are **Nodes** and roads are **Edges**.
2.  **Dijkstra's Algorithm:** Used to calculate the shortest path (minimum time) from an emergency location to all available ambulances.
3.  **Greedy Approach:** Among all reachable ambulances, we locally choose the one that arrives the earliest.

In [19]:
import pandas as pd
import heapq  # We use this for the Priority Queue in Dijkstra's Algorithm
from datetime import datetime, timedelta

# Constants
SPEED_KMPH = 40.0  # Average speed of an ambulance
DATA_DIR = './Datasets' 

print("Libraries imported.")

Libraries imported.


In [20]:
# 1. LOAD DATA
# We load the city map (nodes/edges) and the resources (ambulances)
nodes_df = pd.read_csv(f'{DATA_DIR}/nodes.csv')
edges_df = pd.read_csv(f'{DATA_DIR}/edges.csv')
ambulances_df = pd.read_csv(f'{DATA_DIR}/ambulances.csv')
emergencies_df = pd.read_csv(f'{DATA_DIR}/emergencies.csv')

# Pre-processing:
# We MUST sort emergencies by time. In a real scenario, calls come in chronological order.
emergencies_df['timestamp'] = pd.to_datetime(emergencies_df['timestamp'])
emergencies_df = emergencies_df.sort_values('timestamp').reset_index(drop=True)

print("Data Loaded Successfully.")

Data Loaded Successfully.


C:\Users\user\AppData\Local\Temp\ipykernel_18396\2308217103.py:10: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  emergencies_df['timestamp'] = pd.to_datetime(emergencies_df['timestamp'])


## The Dispatcher Engine

This class encapsulates the core logic. 

**How it works:**
1.  **`__init__`**: Converts the CSV edge list into a Python Dictionary `self.adj`. This acts as our **Adjacency List** for O(1) lookups.
2.  **`dijkstra`**: A standard implementation using a **Min-Heap**. It computes the shortest time from a `start_node` to *every other node* in the city.
3.  **`assign`**: When an emergency occurs:
    * We run Dijkstra from the *Emergency Node* outwards.
    * We check every ambulance to see how far away it is.
    * We calculate `Arrival Time = Current Time + Travel Time`.
    * If an ambulance is busy, we add the Travel Time to its `free_until` time.
    * We pick the ambulance with the **minimum Arrival Time**.

In [21]:
# 2. THE SIMPLIFIED DISPATCHER CLASS
class Dispatcher:
    def __init__(self, edges, ambulances):
        # GRAPH BUILDING:
        # We use a Dictionary for the Adjacency List: { node: [(neighbor, weight), ...] }
        self.adj = {}
        for _, r in edges.iterrows():
            u, v, w = int(r['source_node']), int(r['destination_node']), float(r['distance_km'])
            
            # Undirected Graph: Road goes both ways (u->v and v->u)
            self.adj.setdefault(u, []).append((v, w))
            self.adj.setdefault(v, []).append((u, w))

        # AMBULANCE STATE:
        # We store ambulances as a list of dictionaries to track their status easily.
        self.ambs = []
        for _, r in ambulances.iterrows():
            self.ambs.append({
                'id': int(r['ambulance_id']),
                'loc': int(r['current_node']),
                'free_until': None # None means it is currently sitting idle at its base
            })

    def dijkstra(self, start_node):
        """
        Standard Dijkstra's Algorithm using a Priority Queue (Min-Heap).
        Time Complexity: O(E * log V) where E is edges, V is vertices.
        """
        pq = [(0, start_node)]      # Tuple: (current_dist, node_id)
        dist = {start_node: 0}      # Tracks shortest distance found so far
        parents = {start_node: None} # Used to reconstruct the path later

        while pq:
            d, u = heapq.heappop(pq) # Greedy step: always pop the smallest distance
            
            # Optimization: If we found a shorter way to 'u' before, ignore this old path
            if d > dist.get(u, float('inf')): continue

            # Explore neighbors
            for v, w in self.adj.get(u, []):
                if dist.get(u) + w < dist.get(v, float('inf')):
                    # Relaxation Step: We found a better path!
                    dist[v] = dist[u] + w
                    parents[v] = u
                    heapq.heappush(pq, (dist[v], v))
        return dist, parents

    def assign(self, incident_node, req_time):
        # Step 1: Run Dijkstra from the incident location to find distances to ALL nodes
        dists, parents = self.dijkstra(incident_node)

        best_amb = None
        best_arrival = None
        travel_min = 0

        # Step 2: Iterate through all ambulances to find the "Greedy Best" choice
        for amb in self.ambs:
            if amb['loc'] not in dists: continue # Ambulance is unreachable (island node)

            # Calculate raw travel time
            t_min = (dists[amb['loc']] / SPEED_KMPH) * 60
            
            # Step 3: Check Availability
            # If the ambulance is busy, it can only start AFTER it finishes the current job
            start_time = req_time
            if amb['free_until'] and amb['free_until'] > req_time:
                start_time = amb['free_until']
            
            arrival = start_time + timedelta(minutes=t_min)

            # Step 4: Compare to find the earliest arrival
            if best_amb is None or arrival < best_arrival:
                best_amb = amb
                best_arrival = arrival
                travel_min = t_min

        if best_amb:
            # Step 5: Path Reconstruction (Backtracking from Incident -> Ambulance)
            path = []
            curr = best_amb['loc']
            while curr is not None:
                path.append(curr)
                curr = parents.get(curr)
                if curr == incident_node: 
                    path.append(incident_node)
                    break
            
            # Step 6: Update the Ambulance's state for the NEXT emergency
            start_node = best_amb['loc']
            best_amb['loc'] = incident_node       # Ambulance moves to the scene
            best_amb['free_until'] = best_arrival # Ambulance is now busy until this time
            
            return best_amb['id'], best_arrival, start_node, travel_min, path
        
        return None

## Simulation Loop

Here we process the emergencies one by one. 

**Logic:**
1.  Read the emergency time and location.
2.  Call `system.assign()` to get the best ambulance.
3.  Format the output (Rounding ETA, removing milliseconds).
4.  Store the result.

In [22]:
import time
import tracemalloc  # Built-in library to track memory usage

# 3. MAIN EXECUTION LOOP WITH METRICS
system = Dispatcher(edges_df, ambulances_df)
results = []

print("Starting simulation...")

# --- METRICS START ---
tracemalloc.start()        # Start tracking memory
start_time = time.time()   # Start stopwatch
# ---------------------

for _, row in emergencies_df.iterrows():
    em_id = row['emergency_id']
    node = int(row['location_node'])
    timestamp = row['timestamp']
    urgency = row.get('urgency_level', 'Low')
    
    # Attempt to assign
    assignment = system.assign(node, timestamp)
    
    if assignment:
        amb_id, free_at, start_node, eta, path = assignment
        
        # Formatting
        free_at_str = free_at.strftime('%Y-%m-%d %H:%M:%S')
        eta_rounded = round(eta, 2)
        
        results.append({
            'emergency_id': em_id,
            'urgency_level': urgency,
            'assigned_ambulance': amb_id,
            'ambulance_free_at_what_time': free_at_str,
            'location_node': node,
            'ambulance_start_node': start_node,
            'eta': eta_rounded,
            'status': 'Assigned',
            'path_length_nodes': len(path),
            'path': ",".join(map(str, path))
        })
    else:
        results.append({'emergency_id': em_id, 'status': 'Unassigned', 'eta': None})

# --- METRICS END ---
end_time = time.time()
current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()
# -------------------

# Create DataFrame
out_df = pd.DataFrame(results)

# Calculate Metrics
total_duration = end_time - start_time
total_emergencies = len(emergencies_df)

# 1. Average Response Time (Average of 'eta' column)
avg_response_time = out_df[out_df['status'] == 'Assigned']['eta'].mean()

# 2. Throughput (Emergencies processed / seconds taken)
throughput = total_emergencies / total_duration

# 3. Memory Used (Peak memory in MB)
memory_mb = peak_mem / (1024 * 1024)

# Print Formatting to match your image
print("\n" + "="*30)
print("--- Operational Metrics ---")
print(f"Average Response Time: {avg_response_time:.3f} min")
print(f"Throughput: {throughput:.2f} emergencies/sec")
print(f"Memory Used: {memory_mb:.2f} MB")
print("="*30 + "\n")

out_df.head()

Starting simulation...

--- Operational Metrics ---
Average Response Time: 4.480 min
Throughput: 298.19 emergencies/sec
Memory Used: 0.31 MB



,emergency_id,urgency_level,assigned_ambulance,ambulance_free_at_what_time,location_node,ambulance_start_node,eta,status,path_length_nodes,path
0,454,High,5,2025-01-01 01:16:02,61,8,5.04,Assigned,3,"8,110,61"
1,443,Critical,3,2025-01-01 02:52:23,93,66,10.39,Assigned,4,"66,24,10,93"
2,406,High,14,2025-01-01 04:52:53,26,7,1.89,Assigned,2,"7,26"
3,259,High,13,2025-01-01 05:45:00,52,75,6.01,Assigned,4,"75,34,42,52"
4,334,Low,15,2025-01-01 07:01:25,17,71,3.42,Assigned,4,"71,1,58,17"


In [23]:
# 4. ANALYSIS & SAVING (Exact same output style)
for i in range(min(5, len(out_df))):
    row = out_df.iloc[i]
    print(f"--- Emergency #{row['emergency_id']} Analysis ---")
    print(f"Urgency: {row['urgency_level']}")
    print(f"Status:  {row['status']}")
    if row['status'] == 'Assigned':
        path_str = row['path'].replace(',', ' -> ')
        print(f"Route:   {path_str}")
        print(f"Steps:   {row['path_length_nodes']} nodes visited")
        print(f"Time:    {row['eta']:.2f} minutes")
    print("-" * 40)

out_file = f"simplified_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
out_df.to_csv(out_file, index=False)
print(f"Saved {out_file}")

--- Emergency #454 Analysis ---
Urgency: High
Status:  Assigned
Route:   8 -> 110 -> 61
Steps:   3 nodes visited
Time:    5.04 minutes
----------------------------------------
--- Emergency #443 Analysis ---
Urgency: Critical
Status:  Assigned
Route:   66 -> 24 -> 10 -> 93
Steps:   4 nodes visited
Time:    10.39 minutes
----------------------------------------
--- Emergency #406 Analysis ---
Urgency: High
Status:  Assigned
Route:   7 -> 26
Steps:   2 nodes visited
Time:    1.89 minutes
----------------------------------------
--- Emergency #259 Analysis ---
Urgency: High
Status:  Assigned
Route:   75 -> 34 -> 42 -> 52
Steps:   4 nodes visited
Time:    6.01 minutes
----------------------------------------
--- Emergency #334 Analysis ---
Urgency: Low
Status:  Assigned
Route:   71 -> 1 -> 58 -> 17
Steps:   4 nodes visited
Time:    3.42 minutes
----------------------------------------
Saved simplified_output_20251129_140629.csv
